# 03. Limpieza y creación de indicadores analíticos

En esta etapa transformo las variables originales en un archivo que pueda
analizar cómodamente y, después, conectar a Power BI. La palabra *feature*
aquí significa **indicador descriptivo creado para el análisis**, no variable
de entrenamiento: este proyecto no incluye modelos.

## Decisiones de preparación

| Decisión | Motivo |
|---|---|
| No exportar `SK_ID_CURR` | El dashboard requiere conteos agregados, no identificar solicitudes |
| Convertir `DAYS_BIRTH` a `AGE` | Interpretar años es más claro que días negativos |
| Tratar `DAYS_EMPLOYED=365243` como ausente | Es un valor centinela, no antigüedad real |
| Crear ratios de crédito e ingreso | Comparar exposición relativa y no solo montos absolutos |
| Crear segmentos | Facilitar tablas y filtros de Power BI |

## Carga explícita de las variables que utilizaré

| Función | Qué hace | Para qué la utilizo |
|---|---|---|
| `pd.read_csv(..., usecols=...)` | Lee solo la lista escrita en esta celda | Ver directamente qué campos forman mi análisis |
| `set(...)` | Compara columnas necesarias contra columnas existentes | Confirmar que el CSV es compatible |
| `assert` | Detiene la ejecución si falta alguna variable | No continuar con datos incompletos |

### Lectura línea por línea del código

| Línea o instrucción | Qué estoy haciendo |
|---|---|
| `from pathlib import Path` | Importo la clase que me ayuda a construir rutas. |
| `import numpy as np` | Importo NumPy para usar valores ausentes y condiciones múltiples. |
| `import pandas as pd` | Importo Pandas para cargar y transformar la tabla. |
| `PROJECT_ROOT = ...` | Encuentro la raíz del proyecto sin depender de dónde ejecute el notebook. |
| `DATA_PATH = ...` | Construyo la ruta del CSV fuente. |
| `CLEAN_PATH = ...` | Construyo la ruta del CSV limpio que luego consumirá Power BI. |
| `selected_columns = [...]` | Escribo explícitamente qué variables conservaré. |
| `available_columns = pd.read_csv(..., nrows=0).columns` | Consulto los nombres de columnas reales del archivo. |
| `missing_columns = ...` | Identifico si falta alguna variable solicitada. |
| `assert not missing_columns, ...` | Detengo el proceso si la fuente no tiene la estructura requerida. |
| `raw_data = pd.read_csv(..., usecols=selected_columns)` | Cargo únicamente la información necesaria. |
| `assert set(raw_data["TARGET"]...)...` | Verifico que `TARGET` sea binaria, con valores `0` o `1`. |
| `print(...)` | Informo cuántas filas y columnas cargué. |

In [1]:
# Importo Path para construir rutas del proyecto sin depender del sistema operativo.
from pathlib import Path

# Importo NumPy para valores ausentes y reglas condicionales, y Pandas para tablas.
import numpy as np
import pandas as pd

# Ubico la raíz del proyecto tanto si ejecuto desde ella como desde notebooks/.
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
# Defino la fuente original y el archivo limpio que generaré para Power BI.
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "application_train.csv"
CLEAN_PATH = PROJECT_ROOT / "data" / "clean" / "credit_risk_clean.csv"

# Selecciono las variables necesarias para el análisis descriptivo.
selected_columns = [
    "TARGET", "NAME_CONTRACT_TYPE", "CODE_GENDER", "FLAG_OWN_CAR",
    "FLAG_OWN_REALTY", "CNT_CHILDREN", "AMT_INCOME_TOTAL", "AMT_CREDIT",
    "AMT_ANNUITY", "AMT_GOODS_PRICE", "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS", "NAME_HOUSING_TYPE",
    "DAYS_BIRTH", "DAYS_EMPLOYED", "OCCUPATION_TYPE", "CNT_FAM_MEMBERS",
    "REGION_RATING_CLIENT", "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3",
]
# Leo los encabezados del CSV para confirmar que contiene mis variables elegidas.
available_columns = pd.read_csv(DATA_PATH, nrows=0).columns
# Obtengo la lista de variables requeridas que no aparezcan en el archivo.
missing_columns = sorted(set(selected_columns) - set(available_columns))
# Evito preparar un dataset incompleto si falta alguna columna.
assert not missing_columns, f"Faltan columnas necesarias: {missing_columns}"

# Cargo solo las columnas que voy a limpiar y analizar.
raw_data = pd.read_csv(DATA_PATH, usecols=selected_columns)
# Verifico que TARGET siga representando únicamente dos resultados históricos: 0 y 1.
assert set(raw_data["TARGET"].dropna().unique()).issubset({0, 1}), (
    "TARGET debe contener únicamente 0 y 1."
)
# Muestro cuánta información entrará a mi proceso de preparación.
print(f"Filas cargadas: {len(raw_data):,}; columnas seleccionadas: {raw_data.shape[1]}")

Filas cargadas: 307,511; columnas seleccionadas: 22


## Limpieza y transformación paso a paso

Ahora hago visibles las reglas que producen la tabla limpia. Prefiero que estas
líneas estén en el notebook porque así puedo leer, cambiar y explicar cada
decisión sin depender de un archivo externo.

| Función o expresión | Qué hace | Por qué la aplico |
|---|---|---|
| `.copy()` | Crea una tabla independiente | Conservar intactos los datos recién leídos |
| `.fillna("No informado")` | Completa categorías ausentes | Facilitar filtros en Power BI |
| `.mask(condición)` | Convierte el centinela laboral en nulo | Evitar años laborales irreales |
| `.replace(0, np.nan)` | Cambia ingresos cero por ausente al dividir | Evitar divisiones infinitas |
| `pd.cut()` | Convierte valores numéricos en rangos | Comparar segmentos legibles |
| `np.select()` | Etiqueta perfiles usando condiciones visibles | Describir exposición financiera |

### Lectura línea por línea del código

| Línea o instrucción | Qué estoy haciendo |
|---|---|
| `clean_data = raw_data.copy()` | Creo una copia para limpiar sin modificar mi tabla cargada originalmente. |
| `categorical_columns = [...]` | Reúno las variables de texto que requieren el mismo tratamiento. |
| `for column in categorical_columns:` | Recorro esas variables una a una. |
| `.replace("XNA", np.nan).fillna("No informado")` | Unifico valores especiales y nulos bajo una etiqueta entendible. |
| `employed_days = ...mask(... == 365243)` | Sustituyo el centinela de empleo por nulo para no interpretarlo como años reales. |
| `income_for_ratio = ...replace(0, np.nan)` | Evito dividir entre cero cuando calcule ratios. |
| `clean_data["AGE"] = ...` | Transformo días de nacimiento negativos en edad aproximada en años enteros. |
| `clean_data["YEARS_EMPLOYED"] = ...` | Transformo días laborales en años y evito valores negativos inválidos. |
| `clean_data["CREDIT_INCOME_RATIO"] = ...` | Calculo cuántas veces el crédito representa el ingreso. |
| `clean_data["ANNUITY_INCOME_RATIO"] = ...` | Calculo qué proporción del ingreso representa la cuota anual. |
| `clean_data["TARGET_LABEL"] = ...map(...)` | Creo etiquetas legibles para el resultado histórico. |
| `income_quartiles = ...quantile(...).tolist()` | Obtengo tres puntos de corte que separan ingresos en cuatro grupos. |
| `clean_data["INCOME_SEGMENT"] = pd.cut(...)` | Clasifico el ingreso en bajo, medio bajo, medio alto o alto. |
| `clean_data["AGE_GROUP"] = pd.cut(...)` | Agrupo edades en rangos interpretables para el EDA. |
| `clean_data["CREDIT_INCOME_SEGMENT"] = pd.cut(...)` | Agrupo el ratio crédito/ingreso en intervalos de exposición. |
| `.astype("object").fillna("Sin información")` | Permito asignar texto a ratios faltantes. |
| `lower_income = ... < ...median()` | Marco las solicitudes con ingreso menor a la mediana. |
| `np.select([...], [...], default=...)` | Creo el perfil descriptivo aplicando condiciones en orden. |
| `clean_data.drop(columns=[...])` | Retiro los días originales porque ya creé sus variables interpretables. |
| `print(...)` | Confirmo el tamaño final del dataset preparado. |

In [2]:
# Trabajo sobre una copia para conservar la tabla recién cargada sin cambios.
clean_data = raw_data.copy()

# Agrupo las variables categóricas a las que aplicaré la misma regla de limpieza.
categorical_columns = [
    "NAME_CONTRACT_TYPE", "CODE_GENDER", "FLAG_OWN_CAR", "FLAG_OWN_REALTY",
    "NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS",
    "NAME_HOUSING_TYPE", "OCCUPATION_TYPE",
]
# Recorro cada variable de texto seleccionada.
for column in categorical_columns:
    # Uno categorías desconocidas y nulos bajo la etiqueta 'No informado'.
    clean_data[column] = clean_data[column].replace("XNA", np.nan).fillna("No informado")

# Sustituyo el centinela 365243 por nulo para no calcular una antigüedad irreal.
employed_days = clean_data["DAYS_EMPLOYED"].mask(clean_data["DAYS_EMPLOYED"] == 365243)
# Convierto ingresos iguales a cero en nulos para evitar divisiones infinitas.
income_for_ratio = clean_data["AMT_INCOME_TOTAL"].replace(0, np.nan)

# Transformo los días negativos desde el nacimiento a una edad aproximada en años.
clean_data["AGE"] = (-clean_data["DAYS_BIRTH"] / 365.25).round(0).astype("Int64")
# Transformo los días trabajados válidos a años de experiencia laboral aproximada.
clean_data["YEARS_EMPLOYED"] = (-employed_days / 365.25).clip(lower=0).round(1)
# Mido cuántas veces el crédito solicitado representa el ingreso del cliente.
clean_data["CREDIT_INCOME_RATIO"] = (clean_data["AMT_CREDIT"] / income_for_ratio).round(2)
# Mido qué proporción del ingreso representa la anualidad del crédito.
clean_data["ANNUITY_INCOME_RATIO"] = (clean_data["AMT_ANNUITY"] / income_for_ratio).round(3)
# Cambio los valores 0 y 1 de TARGET por etiquetas legibles para gráficas y Power BI.
clean_data["TARGET_LABEL"] = clean_data["TARGET"].map(
    {0: "Sin dificultad reportada", 1: "Con dificultad de pago"}
)

# Calculo los tres cuartiles que dividirán ingresos en cuatro segmentos observables.
income_quartiles = clean_data["AMT_INCOME_TOTAL"].quantile([0.25, 0.50, 0.75]).tolist()
# Asigno a cada solicitud su segmento relativo de ingreso.
clean_data["INCOME_SEGMENT"] = pd.cut(
    clean_data["AMT_INCOME_TOTAL"],
    bins=[-np.inf, *income_quartiles, np.inf],
    labels=["Bajo", "Medio bajo", "Medio alto", "Alto"],
    include_lowest=True,
)
# Clasifico a cada solicitante en un rango de edad para comparar tasas agregadas.
clean_data["AGE_GROUP"] = pd.cut(
    clean_data["AGE"],
    bins=[0, 25, 35, 45, 55, 65, np.inf],
    labels=["Hasta 25", "26-35", "36-45", "46-55", "56-65", "65+"],
    include_lowest=True,
)
# Agrupo el ratio crédito/ingreso en rangos interpretables de exposición financiera.
clean_data["CREDIT_INCOME_SEGMENT"] = pd.cut(
    clean_data["CREDIT_INCOME_RATIO"],
    bins=[-np.inf, 2, 4, 6, 10, np.inf],
    labels=["Hasta 2x", "2x-4x", "4x-6x", "6x-10x", "Más de 10x"],
    include_lowest=True,
# Convierto la categoría a texto para etiquetar ratios no calculables.
).astype("object").fillna("Sin información")

# Marco qué ingresos están por debajo del valor mediano del dataset.
lower_income = clean_data["AMT_INCOME_TOTAL"] < clean_data["AMT_INCOME_TOTAL"].median()
# Creo perfiles exploratorios aplicando las reglas siguientes en orden de prioridad.
clean_data["ANALYTICAL_PROFILE"] = np.select(
    [
        # Si no puedo calcular ratios, indico falta de información.
        clean_data["CREDIT_INCOME_RATIO"].isna() | clean_data["ANNUITY_INCOME_RATIO"].isna(),
        # Identifico exposición mayor cuando el crédito supera 5 veces un ingreso bajo.
        (clean_data["CREDIT_INCOME_RATIO"] > 5) & lower_income,
        # Identifico exposición intermedia por ratio de crédito o cuota relativa altos.
        (clean_data["CREDIT_INCOME_RATIO"] > 3) | (clean_data["ANNUITY_INCOME_RATIO"] > 0.30),
    ],
    ["Sin información", "Mayor exposición financiera", "Exposición financiera intermedia"],
    default="Menor exposición financiera",
)

# Elimino días originales porque sus versiones en años ya son más interpretables.
clean_data = clean_data.drop(columns=["DAYS_BIRTH", "DAYS_EMPLOYED"])
# Confirmo el volumen de la tabla analítica terminada.
print(f"Filas preparadas: {len(clean_data):,}; columnas analíticas: {clean_data.shape[1]}")

Filas preparadas: 307,511; columnas analíticas: 29


## Indicadores nuevos

| Columna creada | Cálculo o regla | Lectura analítica |
|---|---|---|
| `AGE` | Días de nacimiento / 365,25 en valor positivo | Edad aproximada |
| `YEARS_EMPLOYED` | Días trabajados / 365,25; centinela a nulo | Antigüedad disponible |
| `CREDIT_INCOME_RATIO` | Crédito / ingreso | Cuántas veces el crédito representa el ingreso |
| `ANNUITY_INCOME_RATIO` | Cuota anual / ingreso | Carga relativa de pago |
| `INCOME_SEGMENT` | Cuartiles de ingreso | Comparación de grupos de tamaño parecido |
| `CREDIT_INCOME_SEGMENT` | Rangos del ratio crédito/ingreso | Exposición crediticia relativa |
| `ANALYTICAL_PROFILE` | Reglas de ratios e ingreso | Segmentación exploratoria, no un score |

### Lectura línea por línea del código

| Línea o instrucción | Qué estoy haciendo |
|---|---|
| `indicator_quality = pd.DataFrame({...})` | Creo una tabla de control para los indicadores nuevos. |
| Lista `"indicador"` | Especifico cuáles variables nuevas voy a revisar. |
| Lista `"faltantes"` con `.isna().sum()` | Cuento cuántos valores ausentes tiene cada indicador. |
| Lista `"promedio"` con `.mean()` | Calculo un promedio orientativo de cada indicador. |
| `display(indicator_quality.round(2))` | Muestro el control redondeado a dos decimales. |
| `.value_counts()` | Cuento solicitudes en cada perfil financiero descriptivo. |
| `.rename_axis(...).reset_index(...)` | Presento esos conteos como una tabla legible. |

In [3]:
# Creo una tabla para revisar la calidad de los cuatro indicadores numéricos nuevos.
indicator_quality = pd.DataFrame(
    {
        # Escribo el nombre de cada indicador que quiero auditar.
        "indicador": ["AGE", "YEARS_EMPLOYED", "CREDIT_INCOME_RATIO", "ANNUITY_INCOME_RATIO"],
        # Cuento datos ausentes en cada indicador después de la limpieza.
        "faltantes": [
            clean_data["AGE"].isna().sum(),                 # Edades sin información.
            clean_data["YEARS_EMPLOYED"].isna().sum(),      # Antigüedad ausente o centinela corregido.
            clean_data["CREDIT_INCOME_RATIO"].isna().sum(), # Ratio no calculable por ingreso faltante o cero.
            clean_data["ANNUITY_INCOME_RATIO"].isna().sum(),# Ratio de cuota no calculable.
        ],
        # Calculo el valor promedio de los indicadores para contextualizar la cartera.
        "promedio": [
            clean_data["AGE"].mean(),                 # Edad promedio.
            clean_data["YEARS_EMPLOYED"].mean(),      # Años laborales promedio disponibles.
            clean_data["CREDIT_INCOME_RATIO"].mean(), # Crédito/ingreso promedio.
            clean_data["ANNUITY_INCOME_RATIO"].mean(),# Cuota/ingreso promedio.
        ],
    }
)
# Muestro la auditoría numérica redondeada para facilitar la lectura.
display(indicator_quality.round(2))
# Muestro cuántas solicitudes quedaron en cada perfil descriptivo.
display(clean_data["ANALYTICAL_PROFILE"].value_counts().rename_axis("perfil").reset_index(name="solicitudes"))

,indicador,faltantes,promedio
0,AGE,0,43.91
1,YEARS_EMPLOYED,55374,6.53
2,CREDIT_INCOME_RATIO,0,3.96
3,ANNUITY_INCOME_RATIO,12,0.18


,perfil,solicitudes
0,Menor exposición financiera,140773
1,Exposición financiera intermedia,114598
2,Mayor exposición financiera,52128
3,Sin información,12


## Exportación para Power BI

El CSV limpio se genera localmente en `data/clean/credit_risk_clean.csv`.
Está excluido de Git porque conserva datos a nivel de solicitud. En el
repositorio publicaré únicamente resultados agregados, imágenes y documentación.

| Función | Qué hace | Resultado |
|---|---|---|
| `.mkdir(parents=True, exist_ok=True)` | Crea la carpeta de salida si hace falta | Asegurar que puedo exportar el CSV |
| `.to_csv()` | Guarda la tabla en un archivo separado | Fuente que conectaré a Power BI |

### Lectura línea por línea del código

| Línea o instrucción | Qué estoy haciendo |
|---|---|
| `CLEAN_PATH.parent.mkdir(...)` | Creo la carpeta `data/clean/` si hace falta. |
| `clean_data.to_csv(..., index=False)` | Exporto la tabla preparada sin guardar una columna de índice artificial. |
| `print(...)` de ruta | Confirmo dónde quedó el archivo para conectarlo en Power BI. |
| `print(...)` de protección | Recuerdo que el CSV contiene nivel solicitud y no debe subirse a Git. |

In [4]:
# Creo la carpeta de salida en caso de que aún no exista.
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
# Exporto la tabla preparada sin añadir el índice interno de Pandas.
clean_data.to_csv(CLEAN_PATH, index=False)
# Indico la ruta que usaré después para conectar Power BI.
print(f"Archivo local generado para Power BI: {CLEAN_PATH.relative_to(PROJECT_ROOT)}")
# Recuerdo que el archivo contiene detalle por solicitud y no se versiona.
print("Este archivo está protegido por .gitignore y no debe subirse al repositorio.")

Archivo local generado para Power BI: data\clean\credit_risk_clean.csv
Este archivo está protegido por .gitignore y no debe subirse al repositorio.


### Conclusión parcial

Ya cuento con una tabla analítica reproducible. Un punto importante es que la
segmentación se creó para comparar comportamiento histórico; antes de llamarla
útil para monitoreo debo comprobar si realmente presenta diferencias de tasa.
Eso lo haré en el EDA final.